# Spatial Feature Extraction Pipeline
This notebook handles step 1 & 2 the Data Pipeline & Spatial Feature formulation for the Spatiotemporal Anomaly Detection capstone project. 
It strictly orders files, automatically detects threat phase cutoffs from XML annotation bounding boxes, and generates dense mathematical profiles using an optimized 2D CNN on an Apple Silicon M-series GPU.

In [34]:
import os
import glob
import re
import cv2
import numpy as np
import xml.etree.ElementTree as ET
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense, Input

# 1. Hardware Check - Ensure Metal performance is available 
print("TensorFlow Version:", tf.__version__)
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    print("Apple Silicon GPU Acceleration IS active (tensorflow-metal).")
    for d in physical_devices:
        print(" ->", d)
else:
    print("Warning: No GPU found, operations will fall back to CPU.")

TensorFlow Version: 2.13.0
Apple Silicon GPU Acceleration IS active (tensorflow-metal).
 -> PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [35]:
# 1. Advanced Multi-Key Sorting (Camera ID + Frame Number)
import collections

data_directory = 'Images' # macOS workspace directory

def extract_cam_and_frame(filename):
    """
    Parses both the Camera ID (e.g., Cam1) and the frame sequence id.
    Returns a tuple: (Camera ID string, integer frame number)
    """
    # Uses regex to securely grab the Cam ID and sequence number regardless of other filename string structure
    match = re.search(r'(Cam\d+).*?_frame_(\d+)\.', filename)
    if match:
        return match.group(1), int(match.group(2))
    return "Unknown", -1

# Fetch all jpg paths
all_jpgs = glob.glob(os.path.join(data_directory, '*.jpg'))

# Group paths natively into a dictionary separated strictly by their Camera ID bounds
camera_groups = collections.defaultdict(list)
for p in all_jpgs:
    cam, frame = extract_cam_and_frame(p)
    if cam != "Unknown":
        camera_groups[cam].append(p)

# Sort the individual lists intra-camera by strict frame order logic to preserve temporal continuity
print("--- Raw Sequences Grouped by Camera Stream ---")
for cam in camera_groups:
    camera_groups[cam].sort(key=lambda x: extract_cam_and_frame(x)[1])
    print(f"Discovered {len(camera_groups[cam])} sequential frames bound to {cam}.")

--- Raw Sequences Grouped by Camera Stream ---
Discovered 3511 sequential frames bound to Cam7.
Discovered 607 sequential frames bound to Cam1.
Discovered 1031 sequential frames bound to Cam5.


In [36]:
# 2. Spatiotemporal Separation & Automated XML Filtering per Camera

train_paths = []
test_paths = []

print("\n--- Spatiotemporal Splitting Breakdown ---")
# Process sequentially grouped timelines per camera to prevent cross-contamination
for cam, paths in camera_groups.items():
    cam_train_count = 0
    cam_test_count = 0
    
    for jpg_path in paths:
        # Dynamically locate the corresponding bounding box XML mapping
        xml_path = jpg_path.replace('.jpg', '.xml')
        has_threat = False
        
        # Verify file existence and peek inside exactly for threat boundary parameters limit
        if os.path.exists(xml_path):
            try:
                tree = ET.parse(xml_path)
                root = tree.getroot()
                if len(root.findall('object')) > 0:
                    has_threat = True
            except Exception as e:
                pass # Corrupt XMLs implicitly treated safely (as no threats) or ignored
        
        # 1. TEST LIST: Record full continuous real-time sequences representing narrative streams 
        test_paths.append(jpg_path)
        cam_test_count += 1
        
        # 2. TRAIN LIST: Baseline unsupervised mapping accepts solely weapon-free blank frames 
        if not has_threat:
            train_paths.append(jpg_path)
            cam_train_count += 1
            
    print(f"{cam}: Isolated {cam_train_count} Clean Frames | {cam_test_count} Total Stream Frames")

print(f"\n=================================")
print(f"Consolidated Clean Baseline (Train) Count: {len(train_paths)}")
print(f"Consolidated Real-Time Sequence (Test) Count: {len(test_paths)}")
print(f"=================================")


--- Spatiotemporal Splitting Breakdown ---
Cam7: Isolated 3079 Clean Frames | 3511 Total Stream Frames
Cam1: Isolated 320 Clean Frames | 607 Total Stream Frames
Cam5: Isolated 216 Clean Frames | 1031 Total Stream Frames

Consolidated Clean Baseline (Train) Count: 3615
Consolidated Real-Time Sequence (Test) Count: 5149


In [37]:
# 3. Optimized TensorFlow Dataset Engineering (Dual Target Configuration)

def load_and_preprocess_image(file_path):
    """
    OpenCV transformation pipeline optimized cleanly inside tensor workflow:
    - Target: Grayscale
    - Canvas: Normalize exact 64x64 parameter bounds
    - Value scaling: Scale 0-255 pixels uniformly towards [0.0, 1.0] limits
    """
    image = tf.io.read_file(file_path)
    image = tf.image.decode_jpeg(image, channels=1) # 1-Channel constraint mapping enforcement
    image = tf.image.resize(image, [64, 64]) 
    image = image / 255.0 
    return image

batch_size = 64

# Generate Unsupervised Clean Training Set pipe mapped accurately for optimal VRAM buffer efficiency 
train_dataset = tf.data.Dataset.from_tensor_slices(train_paths)
train_dataset = train_dataset.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Generate Complete Temporal Testing Set pipe 
test_dataset = tf.data.Dataset.from_tensor_slices(test_paths)
test_dataset = test_dataset.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Flatten, Reshape, Conv2DTranspose, UpSampling2D, Input, Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense

print("--- Building the Spatial Autoencoder ---")

# 1. THE ENCODER 
encoder_input = Input(shape=(64, 64, 1), name="Base_Grayscale")
x = Conv2D(32, (3, 3), activation='relu', padding='same')(encoder_input)
x = MaxPooling2D((2, 2))(x)
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D((2, 2))(x)
x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D((2, 2))(x)

# FIX 1: Use Flatten instead of Global Average Pooling to preserve structural locations (8x8x128 = 8192)
x = Flatten()(x)

# FIX 2: Remove 'relu' to prevent dying neurons in the bottleneck
latent_space = Dense(128, name="Spatial_Profile_Embs")(x)

spatial_encoder = Model(encoder_input, latent_space, name="Encoder")

# 2. DEFINE THE DECODER (This teaches the encoder how to retain information)
decoder_input = Input(shape=(128,))
# Reverse the Flatten (8x8x128 = 8192)
x = Dense(8 * 8 * 128, activation='relu')(decoder_input)
x = Reshape((8, 8, 128))(x)
x = Conv2DTranspose(128, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x)
x = Conv2DTranspose(64, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x)
x = Conv2DTranspose(32, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x)
decoder_output = Conv2DTranspose(1, (3, 3), activation='sigmoid', padding='same')(x)

spatial_decoder = Model(decoder_input, decoder_output, name="Decoder")

# 3. COMBINE AND TRAIN
spatial_autoencoder = Model(encoder_input, spatial_decoder(spatial_encoder(encoder_input)))
spatial_autoencoder.compile(optimizer='adam', loss='mse')

print("Training Spatial Autoencoder on Clean Baseline Frames...")
# We train it to reconstruct the safe frames.
# Note: An autoencoder requires (input, target) where target == input.
spatial_autoencoder.fit(train_dataset.map(lambda x: (x, x)), epochs=10) # 10 epochs should be enough and I dont wanan wait

# 4. MATRIX EXPORT (Using the TRAINED encoder)
print("\nInitiating tensor forward-pass mappings using TRAINED weights...")

if len(train_paths) > 0:
    # Notice we use spatial_encoder.predict, NOT spatial_autoencoder.predict
    spatial_features_train = spatial_encoder.predict(train_dataset)
    np.save('spatial_features_train.npy', spatial_features_train)
    print(f"-> Trained Unsupervised Profile Exported: {spatial_features_train.shape}")

if len(test_paths) > 0:
    spatial_features_test = spatial_encoder.predict(test_dataset)
    np.save('spatial_features_test.npy', spatial_features_test)
    print(f"-> Trained Stream Profile Exported: {spatial_features_test.shape}")

--- Building the Spatial Autoencoder ---
Training Spatial Autoencoder on Clean Baseline Frames...
Epoch 1/10


2026-05-20 14:06:10.285184: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


57/57 [==============================] - 4s 56ms/step - loss: 0.0096
Epoch 2/10
57/57 [==============================] - 3s 60ms/step - loss: 0.0087
Epoch 3/10
57/57 [==============================] - 3s 59ms/step - loss: 0.0072
Epoch 4/10
57/57 [==============================] - 3s 58ms/step - loss: 0.0073
Epoch 5/10
57/57 [==============================] - 4s 61ms/step - loss: 0.0069
Epoch 6/10
57/57 [==============================] - 4s 60ms/step - loss: 0.0066
Epoch 7/10
57/57 [==============================] - 3s 59ms/step - loss: 0.0070
Epoch 8/10
57/57 [==============================] - 3s 59ms/step - loss: 0.0070
Epoch 9/10
57/57 [==============================] - 3s 60ms/step - loss: 0.0069
Epoch 10/10
57/57 [==============================] - 3s 59ms/step - loss: 0.0065

Initiating tensor forward-pass mappings using TRAINED weights...
 4/57 [=>............................] - ETA: 1s

2026-05-20 14:06:44.894417: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


57/57 [==============================] - 2s 33ms/step
-> Trained Unsupervised Profile Exported: (3615, 128)
81/81 [==============================] - 3s 32ms/step
-> Trained Stream Profile Exported: (5149, 128)


In [39]:
# Collect and count every single object name found in the entire XML dataset
all_found_objects = {}

# FIX: Define and sort xml_paths so it exists in the notebook namespace
all_xmls = glob.glob(os.path.join(data_directory, '*.xml'))
xml_paths = sorted(all_xmls, key=lambda x: extract_cam_and_frame(x)[1])

for xml_path in xml_paths:
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        for obj in root.findall('object'):
            name_node = obj.find('name')
            if name_node is not None and name_node.text:
                name = name_node.text.strip()
                all_found_objects[name] = all_found_objects.get(name, 0) + 1
    except Exception as e:
        continue

print("=== ALL UNIQUE OBJECT TAGS DISCOVERED IN DATASET ===")
if all_found_objects:
    for obj_name, count in sorted(all_found_objects.items(), key=lambda item: item[1], reverse=True):
        print(f"Tag '{obj_name}': {count} instances found")
else:
    print("No objects found in any XML files.")

=== ALL UNIQUE OBJECT TAGS DISCOVERED IN DATASET ===
Tag 'Handgun': 1714 instances found
Tag 'Short_rifle': 797 instances found
Tag 'Knife': 210 instances found


In [ ]:
#  threat distribution
for i, xml_path in enumerate(xml_paths[:100]): 
    tree = ET.parse(xml_path)
    root = tree.getroot()
    objects = [obj.find('name').text for obj in root.findall('object')]
    if objects:
        # Call the correct function name and extract index 1 for the frame number
        frame_number = extract_cam_and_frame(xml_path)[1]
        print(f"Frame {frame_number} contains: {objects}")

Frame 1 contains: ['Handgun']
Frame 1 contains: ['Handgun', 'Handgun', 'Knife']
Frame 1 contains: ['Short_rifle', 'Handgun', 'Handgun', 'Handgun']
Frame 2 contains: ['Handgun']
Frame 2 contains: ['Handgun']
Frame 2 contains: ['Short_rifle']
Frame 2 contains: ['Handgun', 'Knife', 'Handgun']
Frame 2 contains: ['Knife']
Frame 2 contains: ['Handgun']
Frame 2 contains: ['Handgun']
Frame 2 contains: ['Short_rifle', 'Handgun', 'Handgun']
Frame 2 contains: ['Handgun']
Frame 3 contains: ['Handgun']
Frame 3 contains: ['Handgun']
Frame 3 contains: ['Short_rifle']
Frame 3 contains: ['Handgun', 'Handgun', 'Knife']
Frame 3 contains: ['Handgun']
Frame 3 contains: ['Knife', 'Handgun']
Frame 3 contains: ['Handgun']
Frame 3 contains: ['Handgun', 'Handgun', 'Short_rifle']
Frame 3 contains: ['Handgun']
Frame 4 contains: ['Short_rifle']
Frame 4 contains: ['Handgun']
Frame 4 contains: ['Handgun', 'Handgun']
Frame 4 contains: ['Handgun', 'Knife', 'Handgun']
Frame 4 contains: ['Handgun']
Frame 4 contains: ['K

In [ ]:
# STEP 3: TEMPORAL SEQUENCE ENGINEERING

import numpy as np

def generate_pure_cam_windows(features_matrix, sequence_length=10, stride=1):
    """
    Slices the consolidated feature matrix into sliding windows 
    while strictly respecting camera boundaries to prevent motion flashing.
    """
    # Detect if we are processing the train matrix or the test matrix based on row count
    if features_matrix.shape[0] == 5149:
        camera_limits = [3511, 607, 1031] # Test stream frame allocations per camera
    else:
        camera_limits = [3079, 320, 216]  # Train stream frame allocations per camera
        
    all_sequences = []
    start_idx = 0
    
    # Slice windows independently per camera block
    for limit in camera_limits:
        end_idx = start_idx + limit
        camera_block = features_matrix[start_idx:end_idx]
        
        for i in range(0, len(camera_block) - sequence_length + 1, stride):
            all_sequences.append(camera_block[i:i + sequence_length])
            
        start_idx = end_idx
        
    return np.array(all_sequences)

print("Loading precomputed 2D spatial features...")
features_train = np.load('spatial_features_train.npy')
features_test = np.load('spatial_features_test.npy')

# Generate the 3D Tensors (Sequence Length = 10 frames / 5 seconds of context)
print("\nEngineering spatiotemporal sliding windows...")
X_train_sequences = generate_pure_cam_windows(features_train, sequence_length=10, stride=1)
X_test_sequences = generate_pure_cam_windows(features_test, sequence_length=10, stride=1)

# Export the finalized modeling-ready files
np.save('X_train_sequences.npy', X_train_sequences)
np.save('X_test_sequences.npy', X_test_sequences)

print("\n=== FINAL MODEL-READY TENSORS EXPORTED ===")
print(f"X_train_sequences shape (LSTM Input): {X_train_sequences.shape}") # (Samples, 10, 128)
print(f"X_test_sequences shape (LSTM Input):  {X_test_sequences.shape}")  # (Samples, 10, 128)
print("==========================================")

Loading precomputed 2D spatial features...

Engineering spatiotemporal sliding windows...

=== FINAL MODEL-READY TENSORS EXPORTED ===
X_train_sequences shape (LSTM Input): (3588, 10, 128)
X_test_sequences shape (LSTM Input):  (5122, 10, 128)
